In [6]:
import os
import librosa
import pandas as pd
from pathlib import Path
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle

In [7]:
# === SETUP ===
current_dir = os.getcwd()  # Folder where this notebook runs (EDA/)
project_root = os.path.abspath(os.path.join(current_dir, ".."))  # Go one level up to project root

# Paths relative to project root
audio_root = os.path.join(project_root, "training_data", "training")  # training audio folder
plots_dir = os.path.join(current_dir, "Plots")  # Save plots locally inside EDA/Plots

# Make Plots folder if it does not exist
os.makedirs(plots_dir, exist_ok=True)

# === COLLECT DATA ===
audio_data = []
missing_files = []

for label in os.listdir(audio_root):
    label_path = os.path.join(audio_root, label)
    if not os.path.isdir(label_path):
        continue

    for file in os.listdir(label_path):
        if file.endswith(".wav"):
            file_path = os.path.join(label_path, file)
            try:
                y, sr = librosa.load(file_path, sr=None)
                duration = librosa.get_duration(y=y, sr=sr)
                audio_data.append({"Label": label, "File": file, "Duration": duration})
            except Exception:
                missing_files.append(file)

In [8]:

# === CREATE DATAFRAME (in memory only, not saved) ===
df = pd.DataFrame(audio_data)
output_pdf = os.path.join(plots_dir, "Dataset_Summary.pdf")

# === SUMMARY STATISTICS ===
total_files = len(df)
total_classes = df["Label"].nunique() if total_files > 0 else 0
avg_duration = df["Duration"].mean() if total_files > 0 else 0
files_per_class = df["Label"].value_counts() if total_files > 0 else pd.Series(dtype=int)

# === PRINT SUMMARY ===
print("\nDATASET SUMMARY")
print("----------------")
print(f"Total Audio Files: {total_files}")
print(f"Number of Classes: {total_classes}")
print(f"Average Duration: {avg_duration:.2f} sec\n")
print("Samples per Class:")
print(files_per_class)

if missing_files:
    print("\nMissing or Corrupted Files:")
    for f in missing_files:
        print(" -", f)


DATASET SUMMARY
----------------
Total Audio Files: 2176
Number of Classes: 8
Average Duration: 13.82 sec

Samples per Class:
Label
phonationA    272
phonationE    272
phonationI    272
phonationO    272
phonationU    272
rhythmKA      272
rhythmPA      272
rhythmTA      272
Name: count, dtype: int64


In [9]:
# === PREPARE DATA FOR PDF ===
summary_data = [
    ["Metric", "Value"],
    ["Total Audio Files", total_files],
    ["Number of Classes", total_classes],
    ["Average Duration (sec)", f"{avg_duration:.2f}"],
]

# Add class distribution
summary_data.append(["Samples per Class", ""])
for label, count in files_per_class.items():
    summary_data.append([f"   • {label}", count])

# Add missing files if any
if missing_files:
    summary_data.append(["Missing / Corrupted Files", ""])
    for f in missing_files:
        summary_data.append([f"   - {f}", ""])

# === CREATE PDF REPORT ===
doc = SimpleDocTemplate(output_pdf, pagesize=A4)
styles = getSampleStyleSheet()
story = []

# Title
title_style = ParagraphStyle(
    name="TitleStyle",
    parent=styles["Heading1"],
    alignment=1,
    fontSize=22,
    textColor=colors.HexColor("#1F618D"),
    spaceAfter=20,
)
story.append(Paragraph("Dataset Summary Report", title_style))

# Intro paragraph
intro_text = (
    "This report provides a summary of the audio dataset used for the Dysarthria Detection project. "
    "It includes dataset statistics such as the number of audio samples, class distribution, and average duration."
)
story.append(Paragraph(intro_text, styles["BodyText"]))
story.append(Spacer(1, 20))

# Table
table = Table(summary_data, colWidths=[3*inch, 3.5*inch])
table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1F618D")),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("ALIGN", (0, 0), (-1, -1), "LEFT"),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
    ("FONTSIZE", (0, 0), (-1, 0), 12),
    ("BOTTOMPADDING", (0, 0), (-1, 0), 8),
    ("BACKGROUND", (0, 1), (-1, -1), colors.whitesmoke),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
]))
story.append(table)
story.append(Spacer(1, 20))

# Footer
footer_style = ParagraphStyle(
    name="FooterStyle",
    fontSize=10,
    textColor=colors.HexColor("#7D7D7D"),
    alignment=1,
)
story.append(Paragraph("Generated automatically by the Dysarthria Detection Project.", footer_style))

# === SAVE PDF ===
doc.build(story)
print(f"\nPDF report saved at:\n{output_pdf}")



PDF report saved at:
d:\University\AI\SAND Project\Detecting_Dysarthia\EDA\Plots\Dataset_Summary.pdf
